# PA3 Task 1 on Kaggle (Full FT runs)

Full FT of Qwen3-0.6B needs ~12 GB (FP32 weights + grads + AdamW states), so it does not fit an 8 GB laptop GPU.
Settings: **GPU T4 x1** (or P100), **Internet on**. Use *Save Version -> Save & Run All* so it runs in the background (12 h limit).

Set `METHOD` below to `full_ft` or `full_ft_l2sp` (one per notebook version). Recovery checkpoints (~7 GB each) go to `/tmp`
because `/kaggle/working` is capped at 20 GB; their sizes are still logged. Everything needed for evaluation
(FP16 export, FP32 diagnostic weights, logs) is written to `/kaggle/working/outputs`.

In [ ]:
METHOD = "full_ft"   # or "full_ft_l2sp"
RUN_EVAL = False     # True: also run lm-eval here (adds ~1-2 h per condition on a T4)

In [ ]:
!git clone -q https://github.com/ahmadhassann96/ATML_PA3.git /kaggle/working/ATML_PA3
%cd /kaggle/working/ATML_PA3
!pip install -q "transformers>=4.51" "lm_eval[hf]>=0.4.8" datasets accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
os.environ["PA3_OUT"] = "/kaggle/working/outputs"
os.environ["PA3_CKPT"] = "/tmp/pa3_ckpt"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!python -m task1.data

In [ ]:
!python -m task1.train --method {METHOD}

In [ ]:
if RUN_EVAL:
    conds = ["full_ft@625", "full_ft"] if METHOD == "full_ft" else ["full_ft_l2sp"]
    for c in conds:
        !python -m task1.evaluate --condition {c} --gen_bs 32 --ll_bs 32

In [ ]:
# keep the output small: drop the tokenized data copy (rebuildable) before the version is saved
!rm -rf /kaggle/working/outputs/task1/data/train.npz
!du -sh /kaggle/working/outputs/task1/runs/*

Download `outputs/task1/runs/<METHOD>/` from the version's Output tab and place it at the same path in your local repo,
then run evaluation / diagnostics / qualitative generation locally.